# Notebook 4: OPSD Training on Kaggle (T4 × 2)

This notebook runs the **On-Policy Self-Distillation (OPSD)** condition on GQA using **DistributedDataParallel (DDP)** across both Kaggle T4 GPUs.

### OPSD training condition

For each training example:

1. The **student** sees the full image + question and generates its own answer trajectory.
2. That generated trajectory is frozen.
3. The **student/full-image context** scores the trajectory with gradients enabled.
4. The **teacher/privileged-crop context** scores the exact same trajectory with gradients disabled.
5. Training minimizes token-level **Jensen–Shannon divergence (JSD)** between the two distributions.
6. Only the full-image student branch is optimized.

At inference and validation time, only the **full image** is used.

### Experimental controls

- BERTScore-F1 is the **primary validation metric**
- Normalised exact match is secondary
- best checkpoint selected by validation BERTScore-F1
- same effective batch size target as SFT
- same LoRA rank / alpha / learning-rate family as Notebook 3
- `CE_ANCHOR_WEIGHT = 0.0` for a clean OPSD treatment

### Kaggle dataset structure expected

```text
data/
├── processed/
│   ├── train.csv
│   ├── val.csv
│   ├── test.csv
│   └── all_gqa.csv
├── images/
│   └── <image_id>.jpg
└── crops/
    └── <qa_id>.jpg
```

> **Important:** Enable **GPU T4 ×2** in Kaggle before running this notebook.

3 epochs are run. After every epoch it runs validation, computes **BERTScore-F1** as the primary metric, records normalised exact match as a secondary metric, saves the epoch adapter, and updates `best/` whenever validation BERTScore-F1 improves.

## Install dependencies

In [1]:
%pip uninstall -y torchaudio torchao
%pip install -q -U \
    "transformers>=5.7,<6" \
    "peft>=0.17" \
    "accelerate>=1.0" \
    torchvision \
    pandas \
    Pillow \
    tqdm \
    bottleneck \
    bert-score

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
Note: you may need to restart the kernel to use updated packages.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.4 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


> After the install cell completes, **restart the Kaggle kernel once** if this is a fresh session. This avoids stale imports from Kaggle's preinstalled `torchaudio` / `torchao` packages.

## Verify Kaggle sees both T4 GPUs

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)

    print(
        f"GPU {i}: "
        f"{torch.cuda.get_device_name(i)} | "
        f"{props.total_memory / 1024**3:.1f} GB"
    )

if torch.cuda.device_count() != 2:
    raise RuntimeError(
        "This notebook expects Kaggle GPU T4 x2."
    )


PyTorch: 2.14.1+cu130
CUDA runtime: 13.0
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | 14.6 GB
GPU 1: Tesla T4 | 14.6 GB


## Configure Kaggle paths and experiment settings

Change `KAGGLE_DATASET_SLUG` to the folder name shown under `/kaggle/input/`.

For example, if Kaggle mounts your dataset as:

```text
/kaggle/input/cos802-gqa-data
```

then use:

```python
KAGGLE_DATASET_SLUG = "cos802-gqa-data"
```


In [3]:
from pathlib import Path

# ============================================================
# KAGGLE DATASET
# ============================================================

KAGGLE_DATASET_SLUG = "cos802-gqa-data"

KAGGLE_INPUT = (
    Path("/kaggle/input/datasets/vishalbhagauni/")
    / KAGGLE_DATASET_SLUG
)

DATA = KAGGLE_INPUT
PROCESSED = DATA / "processed"
IMAGES = DATA / "images"
CROPS = DATA / "crops"

WORKING = Path("/kaggle/working")
RESULTS = WORKING / "results"
MODELS_DIR = WORKING / "models"

RESULTS.mkdir(
    parents=True,
    exist_ok=True,
)

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ============================================================
# EXPERIMENT
# ============================================================

MODEL_KEY = "smolvlm"
# "smolvlm", "blip", "minicpm"

SEED = 42

EPOCHS = 3
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.01

PER_DEVICE_BATCH_SIZE = 1

# Effective batch size =
# 1 example × 2 GPUs × 4 accumulation = 8.
GRAD_ACCUM_STEPS = 4

MAX_GRAD_NORM = 1.0

LORA_R = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

MAX_ROLLOUT_TOKENS = 16
MAX_EVAL_TOKENS = 24

DISTILL_TEMPERATURE = 1.0

# OPSD stabilization settings.
FIXED_TEACHER = True
JSD_BETA = 0.5
JSD_TOKEN_CLIP = 1e-6

# Keep this at zero for the primary OPSD experiment.
CE_ANCHOR_WEIGHT = 0.0

# Smoke test suggestion: 32 / 32.
TRAIN_LIMIT = None
VAL_LIMIT = None

SAVE_EVERY_OPT_STEPS = 50

# Validation metric.
BERTSCORE_MODEL = "roberta-large"
BERTSCORE_BATCH_SIZE = 8

print("Dataset:", KAGGLE_INPUT)
print("Processed:", PROCESSED)
print("Images:", IMAGES)
print("Crops:", CROPS)
print("Results:", RESULTS)
print("Models:", MODELS_DIR)

print("Learning rate:", LEARNING_RATE)
print("Fixed teacher:", FIXED_TEACHER)
print("JSD beta:", JSD_BETA)
print("JSD token clip:", JSD_TOKEN_CLIP)


Dataset: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data
Processed: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data/processed
Images: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data/images
Crops: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data/crops
Results: /kaggle/working/results
Models: /kaggle/working/models
Learning rate: 5e-06
Fixed teacher: True
JSD beta: 0.5
JSD token clip: 1e-06


## Validate attached data

In [4]:
import pandas as pd

TRAIN_CSV = PROCESSED / "train.csv"
VAL_CSV = PROCESSED / "val.csv"

for path in [
    TRAIN_CSV,
    VAL_CSV,
]:
    if not path.is_file():
        raise FileNotFoundError(
            f"Missing required file: {path}"
        )

train_check = pd.read_csv(
    TRAIN_CSV,
    nrows=5,
    keep_default_na=False,
)

required_columns = {
    "qa_id",
    "image_id",
    "question",
    "answer",
}

missing = (
    required_columns
    - set(
        train_check.columns
    )
)

if missing:
    raise ValueError(
        f"Missing columns: {sorted(missing)}"
    )

example_image = (
    IMAGES
    / f"{train_check.iloc[0]['image_id']}.jpg"
)

example_crop = (
    CROPS
    / f"{train_check.iloc[0]['qa_id']}.jpg"
)

print("Example image:", example_image)
print("Image exists:", example_image.is_file())

print("Example crop:", example_crop)
print("Crop exists:", example_crop.is_file())

if not example_image.is_file():
    raise FileNotFoundError(
        "Full-image folder does not match the expected structure."
    )

if not example_crop.is_file():
    raise FileNotFoundError(
        "Crop folder does not match the expected structure."
    )


Example image: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data/images/2348621.jpg
Image exists: True
Example crop: /kaggle/input/datasets/vishalbhagauni/cos802-gqa-data/crops/12207280.jpg
Crop exists: True


## Write the two-GPU DDP training script

Training is launched with:

```bash
torchrun --standalone --nproc_per_node=2 train_opsd_ddp.py
```

The teacher/crop forward uses the **unwrapped model under `torch.no_grad()`**, while the student/full-image forward goes through **DDP** so only student gradients participate in distributed reduction.


In [5]:
TRAIN_SCRIPT = r"""
import os
import gc
import json
import random
import re
import shutil
import string
from contextlib import nullcontext
from datetime import timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.distributed as dist
import torch.nn.functional as F

from PIL import Image
from tqdm.auto import tqdm

from torch.nn.parallel import (
    DistributedDataParallel as DDP,
)

from torch.utils.data import (
    Dataset,
    DataLoader,
)

from torch.utils.data.distributed import (
    DistributedSampler,
)

from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BlipProcessor,
    BlipForQuestionAnswering,
)

from peft import (
    LoraConfig,
    get_peft_model,
)

from bert_score import BERTScorer


# ============================================================
# ENVIRONMENT / CONFIG
# ============================================================

KAGGLE_DATASET_SLUG = os.environ[
    "KAGGLE_DATASET_SLUG"
]

DATA = (
    Path("/kaggle/input/datasets/vishalbhagauni/")
    / KAGGLE_DATASET_SLUG
)

PROCESSED = DATA / "processed"
IMAGES = DATA / "images"
CROPS = DATA / "crops"

WORKING = Path("/kaggle/working")
RESULTS = WORKING / "results"
MODELS_DIR = WORKING / "models"


MODEL_KEY = os.environ.get(
    "MODEL_KEY",
    "smolvlm",
)

SEED = int(
    os.environ.get(
        "SEED",
        "42",
    )
)

EPOCHS = int(
    os.environ.get(
        "EPOCHS",
        "3",
    )
)

LEARNING_RATE = float(
    os.environ.get(
        "LEARNING_RATE",
        "5e-6",
    )
)

WEIGHT_DECAY = float(
    os.environ.get(
        "WEIGHT_DECAY",
        "0.01",
    )
)

PER_DEVICE_BATCH_SIZE = int(
    os.environ.get(
        "PER_DEVICE_BATCH_SIZE",
        "1",
    )
)

GRAD_ACCUM_STEPS = int(
    os.environ.get(
        "GRAD_ACCUM_STEPS",
        "4",
    )
)

MAX_GRAD_NORM = float(
    os.environ.get(
        "MAX_GRAD_NORM",
        "1.0",
    )
)

LORA_R = int(
    os.environ.get(
        "LORA_R",
        "8",
    )
)

LORA_ALPHA = int(
    os.environ.get(
        "LORA_ALPHA",
        "16",
    )
)

LORA_DROPOUT = float(
    os.environ.get(
        "LORA_DROPOUT",
        "0.05",
    )
)

MAX_ROLLOUT_TOKENS = int(
    os.environ.get(
        "MAX_ROLLOUT_TOKENS",
        "16",
    )
)

MAX_EVAL_TOKENS = int(
    os.environ.get(
        "MAX_EVAL_TOKENS",
        "24",
    )
)

DISTILL_TEMPERATURE = float(
    os.environ.get(
        "DISTILL_TEMPERATURE",
        "1.0",
    )
)

FIXED_TEACHER = (
    os.environ.get(
        "FIXED_TEACHER",
        "1",
    )
    == "1"
)

JSD_BETA = float(
    os.environ.get(
        "JSD_BETA",
        "0.5",
    )
)

JSD_TOKEN_CLIP = float(
    os.environ.get(
        "JSD_TOKEN_CLIP",
        "1e-6",
    )
)

CE_ANCHOR_WEIGHT = float(
    os.environ.get(
        "CE_ANCHOR_WEIGHT",
        "0.0",
    )
)

if CE_ANCHOR_WEIGHT != 0.0:
    raise ValueError(
        "This Kaggle notebook is configured for the primary "
        "pure-OPSD experiment. Keep CE_ANCHOR_WEIGHT = 0.0."
    )

SAVE_EVERY_OPT_STEPS = int(
    os.environ.get(
        "SAVE_EVERY_OPT_STEPS",
        "50",
    )
)

BERTSCORE_MODEL = os.environ.get(
    "BERTSCORE_MODEL",
    "roberta-large",
)

BERTSCORE_BATCH_SIZE = int(
    os.environ.get(
        "BERTSCORE_BATCH_SIZE",
        "8",
    )
)

TRAIN_LIMIT_RAW = os.environ.get(
    "TRAIN_LIMIT",
    "",
)

VAL_LIMIT_RAW = os.environ.get(
    "VAL_LIMIT",
    "",
)

TRAIN_LIMIT = (
    int(
        TRAIN_LIMIT_RAW
    )
    if TRAIN_LIMIT_RAW
    else None
)

VAL_LIMIT = (
    int(
        VAL_LIMIT_RAW
    )
    if VAL_LIMIT_RAW
    else None
)


MODEL_IDS = {
    "smolvlm":
        "HuggingFaceTB/SmolVLM-500M-Instruct",

    "blip":
        "Salesforce/blip-vqa-base",

    "minicpm":
        "openbmb/MiniCPM-V-4.6",
}

MODEL_ID = MODEL_IDS[
    MODEL_KEY
]


# ============================================================
# DDP INITIALIZATION
# ============================================================

LOCAL_RANK = int(
    os.environ["LOCAL_RANK"]
)

torch.cuda.set_device(
    LOCAL_RANK
)

DEVICE = torch.device(
    f"cuda:{LOCAL_RANK}"
)

MODEL_DTYPE = torch.float16

# Rank 1 waits while rank 0 performs full validation +
# BERTScore. Use a longer timeout than NCCL's default.
dist.init_process_group(
    backend="nccl",
    device_id=DEVICE,
    timeout=timedelta(
        hours=1
    ),
)

RANK = dist.get_rank()
WORLD_SIZE = dist.get_world_size()


def is_main():
    return RANK == 0


def rank0_print(
    *args,
    **kwargs,
):
    if is_main():
        print(
            *args,
            **kwargs,
            flush=True,
        )


# ============================================================
# REPRODUCIBILITY
# ============================================================

rank_seed = (
    SEED
    + RANK
)

random.seed(
    rank_seed
)

np.random.seed(
    rank_seed
)

torch.manual_seed(
    rank_seed
)

torch.cuda.manual_seed_all(
    rank_seed
)


# ============================================================
# OUTPUT PATHS
# ============================================================

RUN_DIR = (
    RESULTS
    / "training"
    / "opsd"
    / MODEL_KEY
    / f"seed{SEED}"
)

ADAPTER_DIR = (
    MODELS_DIR
    / "opsd"
    / MODEL_KEY
    / f"seed{SEED}"
)

LATEST_DIR = (
    ADAPTER_DIR
    / "latest"
)

BEST_DIR = (
    ADAPTER_DIR
    / "best"
)

if is_main():
    RUN_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    ADAPTER_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

dist.barrier()


# ============================================================
# LOAD / REMAP DATA
# ============================================================

train_df = pd.read_csv(
    PROCESSED
    / "train.csv",
    keep_default_na=False,
)

val_df = pd.read_csv(
    PROCESSED
    / "val.csv",
    keep_default_na=False,
)


def remap_paths(
    df,
):
    df = df.copy()

    # Preserve original filenames created by Notebook 01.
    # This is essential for zero-padded crops such as
    # 00213619.jpg, which cannot safely be reconstructed
    # from a numeric qa_id.

    if "image_path" in df.columns:

        df[
            "image_path"
        ] = (
            df[
                "image_path"
            ]
            .astype(str)
            .map(
                lambda path:
                    str(
                        IMAGES
                        / Path(path).name
                    )
            )
        )

    else:

        df[
            "image_path"
        ] = (
            df[
                "image_id"
            ]
            .astype(str)
            .map(
                lambda image_id:
                    str(
                        IMAGES
                        / f"{image_id}.jpg"
                    )
            )
        )


    if "crop_path" not in df.columns:

        raise ValueError(
            "Expected crop_path in the processed GQA CSV. "
            "The original crop filename is required so "
            "zero-padding is preserved."
        )


    df[
        "crop_path"
    ] = (
        df[
            "crop_path"
        ]
        .astype(str)
        .map(
            lambda path:
                str(
                    CROPS
                    / Path(path).name
                )
        )
    )


    return df


train_df = remap_paths(
    train_df
)

val_df = remap_paths(
    val_df
)


def validate_local_paths(
    df,
    split_name,
):
    missing_images = [
        path
        for path in df[
            "image_path"
        ].tolist()
        if not Path(
            path
        ).is_file()
    ]

    missing_crops = [
        path
        for path in df[
            "crop_path"
        ].tolist()
        if not Path(
            path
        ).is_file()
    ]

    if missing_images:
        raise FileNotFoundError(
            f"{split_name}: "
            f"{len(missing_images)} full images are missing. "
            f"First missing path: {missing_images[0]}"
        )

    if missing_crops:
        raise FileNotFoundError(
            f"{split_name}: "
            f"{len(missing_crops)} crops are missing. "
            f"First missing path: {missing_crops[0]}"
        )


validate_local_paths(
    train_df,
    "train",
)

validate_local_paths(
    val_df,
    "validation",
)


if TRAIN_LIMIT is not None:
    train_df = (
        train_df
        .sample(
            n=min(
                TRAIN_LIMIT,
                len(
                    train_df
                ),
            ),
            random_state=SEED,
        )
        .reset_index(
            drop=True
        )
    )

if VAL_LIMIT is not None:
    val_df = (
        val_df
        .sample(
            n=min(
                VAL_LIMIT,
                len(
                    val_df
                ),
            ),
            random_state=SEED,
        )
        .reset_index(
            drop=True
        )
    )


rank0_print(
    "World size:",
    WORLD_SIZE,
)

rank0_print(
    "Train examples:",
    len(
        train_df
    ),
)

rank0_print(
    "Validation examples:",
    len(
        val_df
    ),
)

rank0_print(
    "Effective batch size:",
    (
        PER_DEVICE_BATCH_SIZE
        * WORLD_SIZE
        * GRAD_ACCUM_STEPS
    ),
)


# ============================================================
# DATASET / DISTRIBUTED SAMPLER
# ============================================================

class GQADataset(
    Dataset
):

    def __init__(
        self,
        df,
    ):
        self.df = (
            df
            .reset_index(
                drop=True
            )
        )

    def __len__(
        self,
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        idx,
    ):
        row = self.df.iloc[
            idx
        ]

        return {
            "qa_id":
                str(
                    row.qa_id
                ),

            "image_path":
                str(
                    row.image_path
                ),

            "crop_path":
                str(
                    row.crop_path
                ),

            "question":
                str(
                    row.question
                ),

            "answer":
                str(
                    row.answer
                ),
        }


def collate_rows(
    batch,
):
    return batch


train_dataset = GQADataset(
    train_df
)

train_sampler = DistributedSampler(
    train_dataset,
    num_replicas=WORLD_SIZE,
    rank=RANK,
    shuffle=True,
    seed=SEED,
    drop_last=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=PER_DEVICE_BATCH_SIZE,
    sampler=train_sampler,
    num_workers=2,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_rows,
)


# ============================================================
# LOAD MODEL
# ============================================================

rank0_print(
    "Loading model:",
    MODEL_ID,
)


if MODEL_KEY == "blip":

    processor = (
        BlipProcessor
        .from_pretrained(
            MODEL_ID
        )
    )

    base_model = (
        BlipForQuestionAnswering
        .from_pretrained(
            MODEL_ID,
            dtype=MODEL_DTYPE,
        )
    )

else:

    processor = (
        AutoProcessor
        .from_pretrained(
            MODEL_ID
        )
    )

    base_model = (
        AutoModelForImageTextToText
        .from_pretrained(
            MODEL_ID,
            dtype=MODEL_DTYPE,
            low_cpu_mem_usage=True,
        )
    )


# BLIP produced autograd/in-place errors when gradient
# checkpointing was enabled in the SFT run.
if MODEL_KEY != "blip":

    if hasattr(
        base_model,
        "gradient_checkpointing_enable",
    ):
        try:
            base_model.gradient_checkpointing_enable()

        except Exception:
            pass


if hasattr(
    base_model.config,
    "use_cache",
):
    base_model.config.use_cache = False


# ============================================================
# MODEL-SPECIFIC LORA TARGETS
# ============================================================

if MODEL_KEY == "blip":

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules=[
            "query",
            "value",
        ],
    )

else:

    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        target_modules="all-linear",
    )


base_model = get_peft_model(
    base_model,
    lora_config,
)

base_model = base_model.to(
    DEVICE
)


# MiniCPM has trainable LoRA parameters that may not
# participate in every forward path.
FIND_UNUSED_PARAMETERS = (
    MODEL_KEY
    == "minicpm"
)

model = DDP(
    base_model,
    device_ids=[
        LOCAL_RANK
    ],
    output_device=LOCAL_RANK,
    find_unused_parameters=(
        FIND_UNUSED_PARAMETERS
    ),
)


if is_main():
    base_model.print_trainable_parameters()


def teacher_adapter_context():
    # Fixed-teacher OPSD with PEFT:
    # base pretrained weights stay frozen; teacher scoring
    # disables the student's LoRA adapter.
    if not FIXED_TEACHER:
        return nullcontext()

    if not hasattr(
        model.module,
        "disable_adapter",
    ):
        raise RuntimeError(
            "FIXED_TEACHER=True requires PEFT disable_adapter()."
        )

    return model.module.disable_adapter()


# ============================================================
# IMAGE / BATCH HELPERS
# ============================================================

def open_rgb(
    path,
):
    with Image.open(
        path
    ) as image:
        return image.convert(
            "RGB"
        )


def move_batch(
    batch,
):
    moved = {}

    for key, value in batch.items():

        if not torch.is_tensor(
            value
        ):
            moved[
                key
            ] = value

            continue

        if (
            key
            in {
                "pixel_values",
                "pixel_values_videos",
            }
            and value.is_floating_point()
        ):
            moved[
                key
            ] = value.to(
                device=DEVICE,
                dtype=MODEL_DTYPE,
                non_blocking=True,
            )

        else:
            moved[
                key
            ] = value.to(
                DEVICE,
                non_blocking=True,
            )

    return moved


def chat_kwargs():

    if MODEL_KEY == "minicpm":

        return {
            "enable_thinking":
                False,

            "processor_kwargs":
                {
                    "downsample_mode":
                        "4x",

                    "max_slice_nums":
                        1,
                },
        }

    return {}


# ============================================================
# STUDENT ON-POLICY ROLLOUT
# ============================================================

@torch.inference_mode()
def rollout_student(
    image_path,
    question,
    max_tokens=MAX_ROLLOUT_TOKENS,
):
    # Generation does not need DDP reduction hooks.
    generation_model = (
        model.module
    )

    was_training = (
        generation_model.training
    )

    generation_model.eval()

    image = open_rgb(
        image_path
    )


    if MODEL_KEY == "blip":

        inputs = processor(
            images=image,
            text=str(
                question
            ),
            return_tensors="pt",
        )

        inputs = move_batch(
            inputs
        )

        generated = (
            generation_model
            .generate(
                **inputs,
                max_new_tokens=max_tokens,
                do_sample=False,
            )
        )

        text = (
            processor
            .decode(
                generated[
                    0
                ],
                skip_special_tokens=True,
            )
            .strip()
        )

        rollout_ids = (
            generated.detach()
        )


    else:

        messages = [
            {
                "role":
                    "user",

                "content":
                    [
                        {
                            "type":
                                "image",

                            "image":
                                image,
                        },
                        {
                            "type":
                                "text",

                            "text":
                                str(
                                    question
                                )
                                + "\nAnswer briefly with only the answer.",
                        },
                    ],
            }
        ]

        inputs = (
            processor
            .apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
                **chat_kwargs(),
            )
        )

        inputs = move_batch(
            inputs
        )

        generation_kwargs = {
            "max_new_tokens":
                max_tokens,

            "do_sample":
                False,
        }

        if MODEL_KEY == "minicpm":

            generation_kwargs[
                "downsample_mode"
            ] = "4x"


        generated = (
            generation_model
            .generate(
                **inputs,
                **generation_kwargs,
            )
        )

        trimmed = [
            out_ids[
                len(
                    in_ids
                ):
            ]

            for in_ids, out_ids
            in zip(
                inputs[
                    "input_ids"
                ],
                generated,
            )
        ]

        text = (
            processor
            .batch_decode(
                trimmed,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=False,
            )[
                0
            ]
            .strip()
        )

        rollout_ids = (
            trimmed[
                0
            ]
            .detach()
        )


    if was_training:
        generation_model.train()

    return (
        text,
        rollout_ids,
    )


# ============================================================
# SCORE FIXED TRAJECTORY
# ============================================================

def chat_rollout_logits(
    image_path,
    question,
    rollout_ids,
    student_branch,
):
    # Score the exact generated token trajectory. There is no
    # decode -> re-tokenize round trip for SmolVLM/MiniCPM.

    image = open_rgb(
        image_path
    )

    prompt = [
        {
            "role":
                "user",

            "content":
                [
                    {
                        "type":
                            "image",

                        "image":
                            image,
                    },
                    {
                        "type":
                            "text",

                        "text":
                            str(
                                question
                            )
                            + "\nAnswer briefly with only the answer.",
                    },
                ],
        }
    ]


    inputs = (
        processor
        .apply_chat_template(
            prompt,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt",
            **chat_kwargs(),
        )
    )

    inputs = move_batch(
        inputs
    )


    rollout_ids = (
        rollout_ids
        .reshape(
            1,
            -1,
        )
        .to(
            DEVICE
        )
    )


    if rollout_ids.numel() == 0:
        raise ValueError(
            "Student generated zero rollout tokens."
        )


    prompt_length = (
        inputs[
            "input_ids"
        ].shape[
            1
        ]
    )


    inputs[
        "input_ids"
    ] = torch.cat(
        [
            inputs[
                "input_ids"
            ],
            rollout_ids,
        ],
        dim=1,
    )


    if "attention_mask" in inputs:

        rollout_mask = torch.ones(
            (
                inputs[
                    "attention_mask"
                ].shape[
                    0
                ],
                rollout_ids.shape[
                    1
                ],
            ),
            dtype=inputs[
                "attention_mask"
            ].dtype,
            device=DEVICE,
        )

        inputs[
            "attention_mask"
        ] = torch.cat(
            [
                inputs[
                    "attention_mask"
                ],
                rollout_mask,
            ],
            dim=1,
        )


    if student_branch:

        if MODEL_KEY == "minicpm":

            outputs = model(
                **inputs,
                downsample_mode="4x",
            )

        else:

            outputs = model(
                **inputs
            )

    else:

        with torch.no_grad(), teacher_adapter_context():

            if MODEL_KEY == "minicpm":

                outputs = model.module(
                    **inputs,
                    downsample_mode="4x",
                )

            else:

                outputs = model.module(
                    **inputs
                )


    # For a causal decoder, logits at position t-1 predict
    # the token at position t.
    start_index = (
        prompt_length
        - 1
    )

    end_index = (
        start_index
        + rollout_ids.shape[
            1
        ]
    )

    logits = (
        outputs.logits[
            0,
            start_index:end_index,
            :
        ]
    )


    if logits.shape[
        0
    ] != rollout_ids.shape[
        1
    ]:

        raise RuntimeError(
            "Rollout/logit length mismatch: "
            f"rollout={rollout_ids.shape[1]}, "
            f"logits={logits.shape[0]}"
        )


    return (
        logits,
        rollout_ids[
            0
        ],
    )


def blip_answer_logits(
    image_path,
    question,
    rollout_ids,
    student_branch,
):
    image = open_rgb(
        image_path
    )

    question_inputs = processor(
        images=image,
        text=str(
            question
        ),
        return_tensors="pt",
    )

    question_inputs = move_batch(
        question_inputs
    )

    sequence = rollout_ids

    if sequence.ndim == 1:
        sequence = sequence.unsqueeze(
            0
        )

    sequence = sequence.to(
        DEVICE
    )

    if sequence.shape[1] < 2:
        raise ValueError(
            "BLIP rollout is too short for OPSD scoring."
        )

    decoder_input_ids = sequence[:, :-1]
    target_ids = sequence[:, 1:]

    active_model = model.module

    adapter_context = (
        nullcontext()
        if student_branch
        else teacher_adapter_context()
    )

    grad_context = (
        nullcontext()
        if student_branch
        else torch.no_grad()
    )

    with grad_context, adapter_context:

        blip_model = (
            active_model
            .base_model
            .model
        )

        vision_outputs = (
            blip_model
            .vision_model(
                pixel_values=question_inputs[
                    "pixel_values"
                ],
                return_dict=True,
            )
        )

        image_embeds = (
            vision_outputs.last_hidden_state
        )

        image_attention_mask = torch.ones(
            image_embeds.size()[:-1],
            dtype=torch.long,
            device=DEVICE,
        )

        question_outputs = (
            blip_model
            .text_encoder(
                input_ids=question_inputs[
                    "input_ids"
                ],
                attention_mask=question_inputs.get(
                    "attention_mask"
                ),
                encoder_hidden_states=image_embeds,
                encoder_attention_mask=image_attention_mask,
                return_dict=True,
            )
        )

        decoder_outputs = (
            blip_model
            .text_decoder(
                input_ids=decoder_input_ids,
                encoder_hidden_states=question_outputs.last_hidden_state,
                encoder_attention_mask=question_inputs.get(
                    "attention_mask"
                ),
                return_dict=True,
            )
        )

        logits = decoder_outputs.logits

    length = min(
        logits.shape[1],
        target_ids.shape[1],
    )

    if length == 0:
        raise ValueError(
            "BLIP produced zero decoder positions for OPSD."
        )

    return (
        logits[0, :length, :],
        target_ids[0, :length],
    )


def answer_logits(
    image_path,
    question,
    rollout_text,
    rollout_ids,
    student_branch,
):

    if MODEL_KEY == "blip":

        return blip_answer_logits(
            image_path,
            question,
            rollout_ids,
            student_branch,
        )

    return chat_rollout_logits(
        image_path,
        question,
        rollout_ids,
        student_branch,
    )


# ============================================================
# TOKEN-LEVEL JENSEN-SHANNON DIVERGENCE
# ============================================================

def jsd_from_logits(
    student_logits,
    teacher_logits,
    temperature=DISTILL_TEMPERATURE,
    beta=JSD_BETA,
    token_clip=JSD_TOKEN_CLIP,
):
    token_count = min(
        student_logits.shape[0],
        teacher_logits.shape[0],
    )

    if token_count == 0:
        raise ValueError(
            "No rollout tokens available for JSD."
        )

    student_logits = (
        student_logits[:token_count]
        .float()
        / temperature
    )

    teacher_logits = (
        teacher_logits[:token_count]
        .detach()
        .float()
        / temperature
    )

    if not torch.isfinite(student_logits).all():
        raise FloatingPointError(
            "Student logits contain NaN or Inf."
        )

    if not torch.isfinite(teacher_logits).all():
        raise FloatingPointError(
            "Teacher logits contain NaN or Inf."
        )

    student_log_probs = F.log_softmax(
        student_logits,
        dim=-1,
    )

    teacher_log_probs = F.log_softmax(
        teacher_logits,
        dim=-1,
    )

    beta_tensor = torch.tensor(
        beta,
        dtype=student_log_probs.dtype,
        device=student_log_probs.device,
    )

    mixture_log_probs = torch.logsumexp(
        torch.stack(
            [
                student_log_probs
                + torch.log1p(-beta_tensor),

                teacher_log_probs
                + torch.log(beta_tensor),
            ],
            dim=0,
        ),
        dim=0,
    )

    kl_teacher = F.kl_div(
        mixture_log_probs,
        teacher_log_probs,
        reduction="none",
        log_target=True,
    )

    kl_student = F.kl_div(
        mixture_log_probs,
        student_log_probs,
        reduction="none",
        log_target=True,
    )

    pointwise_jsd = (
        beta_tensor * kl_teacher
        + (1.0 - beta_tensor) * kl_student
    )

    # Clip each vocabulary-coordinate contribution before
    # summing over vocabulary.
    if token_clip is not None and token_clip > 0.0:
        pointwise_jsd = pointwise_jsd.clamp(
            max=token_clip
        )

    token_jsd = pointwise_jsd.sum(
        dim=-1
    )

    loss = (
        token_jsd.mean()
        * (temperature ** 2)
    )

    if not torch.isfinite(loss):
        raise FloatingPointError(
            "Clipped JSD became NaN or Inf."
        )

    return loss


# ============================================================
# VALIDATION HELPERS
# ============================================================

ARTICLES = re.compile(
    r"\b(a|an|the)\b"
)

PUNCT = str.maketrans(
    "",
    "",
    string.punctuation,
)


def normalize_answer(
    value,
):
    text = (
        str(
            value
        )
        .lower()
        .translate(
            PUNCT
        )
    )

    text = ARTICLES.sub(
        " ",
        text,
    )

    return " ".join(
        text.split()
    )


@torch.inference_mode()
def predict_answer(
    image_path,
    question,
):
    prediction, _ = rollout_student(
        image_path,
        question,
        max_tokens=MAX_EVAL_TOKENS,
    )

    return prediction


def validate_epoch(
    epoch,
):
    # Rank 0 only.
    model.module.eval()

    records = []

    for row in tqdm(
        val_df.itertuples(
            index=False
        ),
        total=len(
            val_df
        ),
        desc=(
            f"OPSD validation "
            f"epoch {epoch}"
        ),
    ):

        try:
            prediction = predict_answer(
                row.image_path,
                str(
                    row.question
                ),
            )

            error = ""

        except Exception as exc:
            prediction = ""

            error = (
                f"{type(exc).__name__}: "
                f"{str(exc)[:300]}"
            )

        records.append(
            {
                "qa_id":
                    str(
                        row.qa_id
                    ),

                "question":
                    str(
                        row.question
                    ),

                "answer":
                    str(
                        row.answer
                    ),

                "prediction":
                    prediction,

                "error":
                    error,
            }
        )


    predictions = pd.DataFrame(
        records
    )

    valid = (
        predictions.error.eq("")
        & predictions.prediction
        .astype(str)
        .str.strip()
        .ne("")
    )

    evaluated = (
        predictions[
            valid
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    if len(
        evaluated
    ) == 0:
        raise RuntimeError(
            "No valid predictions were produced during validation."
        )


    evaluated[
        "exact_match"
    ] = (
        evaluated.prediction
        .map(
            normalize_answer
        )
        .eq(
            evaluated.answer
            .map(
                normalize_answer
            )
        )
    )

    val_exact_match = float(
        evaluated.exact_match.mean()
    )


    # Load only for scoring this epoch, then release it before
    # returning to OPSD training.
    scorer = BERTScorer(
        model_type=BERTSCORE_MODEL,
        lang="en",
        device=str(
            DEVICE
        ),
        batch_size=BERTSCORE_BATCH_SIZE,
    )

    bert_p, bert_r, bert_f1 = scorer.score(
        evaluated.prediction
        .astype(str)
        .tolist(),
        evaluated.answer
        .astype(str)
        .tolist(),
    )

    evaluated[
        "bertscore_precision"
    ] = bert_p.cpu().numpy()

    evaluated[
        "bertscore_recall"
    ] = bert_r.cpu().numpy()

    evaluated[
        "bertscore_f1"
    ] = bert_f1.cpu().numpy()

    val_bertscore_f1 = float(
        evaluated.bertscore_f1.mean()
    )


    predictions.to_csv(
        RUN_DIR
        / f"val_epoch{epoch}_predictions.csv",
        index=False,
    )

    evaluated.to_csv(
        RUN_DIR
        / f"val_epoch{epoch}_predictions_scored.csv",
        index=False,
    )


    del scorer
    del bert_p
    del bert_r
    del bert_f1

    gc.collect()

    torch.cuda.empty_cache()


    return (
        val_bertscore_f1,
        val_exact_match,
    )


# ============================================================
# OPTIMIZER
# ============================================================

optimizer = torch.optim.AdamW(
    [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


# ============================================================
# OPSD TRAINING
# ============================================================

history = []

optimizer_step = 0

best_bertscore_f1 = float(
    "-inf"
)

best_epoch = None


optimizer.zero_grad(
    set_to_none=True
)


for epoch in range(
    1,
    EPOCHS + 1,
):
    train_sampler.set_epoch(
        epoch
    )

    model.train()

    local_loss_sum = 0.0
    local_examples = 0

    progress = tqdm(
        train_loader,
        total=len(
            train_loader
        ),
        desc=(
            f"OPSD epoch "
            f"{epoch}/{EPOCHS}"
        ),
        disable=not is_main(),
    )


    total_batches = len(
        train_loader
    )


    for batch_index, batch_rows in enumerate(
        progress,
        start=1,
    ):
        # We deliberately fail fast on DDP errors rather than
        # skipping independently on one rank and desynchronizing
        # distributed gradient reduction.
        should_sync = (
            batch_index
            % GRAD_ACCUM_STEPS
            == 0
            or batch_index
            == total_batches
        )

        sync_context = (
            nullcontext()
            if should_sync
            else model.no_sync()
        )


        with sync_context:

            batch_loss = None
            successful = 0


            for row in batch_rows:

                try:

                    # ----------------------------------------
                    # 1. Student on-policy rollout: full image
                    # ----------------------------------------

                    (
                        rollout_text,
                        rollout_ids,
                    ) = rollout_student(
                        row[
                            "image_path"
                        ],
                        row[
                            "question"
                        ],
                    )


                    # An EOS/special-token-only trajectory may decode
                    # to an empty string but is still a valid on-policy
                    # trajectory. Only a genuinely zero-token rollout fails.
                    if rollout_ids.numel() == 0:
                        raise RuntimeError(
                            "Student generated zero rollout tokens."
                        )


                    # Restore train mode for the scoring passes.
                    model.train()


                    # ----------------------------------------
                    # 2. Privileged teacher: crop, no gradients
                    # ----------------------------------------

                    (
                        teacher_logits,
                        _,
                    ) = answer_logits(
                        row[
                            "crop_path"
                        ],
                        row[
                            "question"
                        ],
                        rollout_text,
                        rollout_ids,
                        student_branch=False,
                    )

                    teacher_logits = (
                        teacher_logits.detach()
                    )


                    # ----------------------------------------
                    # 3. Student: full image, DDP gradients
                    # ----------------------------------------

                    (
                        student_logits,
                        _,
                    ) = answer_logits(
                        row[
                            "image_path"
                        ],
                        row[
                            "question"
                        ],
                        rollout_text,
                        rollout_ids,
                        student_branch=True,
                    )


                    # ----------------------------------------
                    # 4. OPSD JSD objective
                    # ----------------------------------------

                    example_loss = (
                        jsd_from_logits(
                            student_logits,
                            teacher_logits,
                        )
                    )


                    if not torch.isfinite(
                        example_loss
                    ):
                        raise FloatingPointError(
                            "OPSD example loss is NaN or Inf."
                        )


                    if batch_loss is None:
                        batch_loss = (
                            example_loss
                        )

                    else:
                        batch_loss = (
                            batch_loss
                            + example_loss
                        )


                    local_loss_sum += float(
                        example_loss
                        .detach()
                        .float()
                        .cpu()
                    )

                    local_examples += 1
                    successful += 1


                    del teacher_logits
                    del student_logits


                except Exception as exc:

                    raise RuntimeError(
                        f"OPSD failed on QA "
                        f"{row['qa_id']} "
                        f"on rank {RANK}: "
                        f"{type(exc).__name__}: "
                        f"{str(exc)}"
                    ) from exc


            if successful == 0:
                raise RuntimeError(
                    "A DDP batch contained no successful examples."
                )


            batch_loss = (
                batch_loss
                / successful
            )


            scaled_loss = (
                batch_loss
                / GRAD_ACCUM_STEPS
            )


            scaled_loss.backward()


        if should_sync:

            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p in model.parameters()
                    if p.requires_grad
                ],
                MAX_GRAD_NORM,
            )

            optimizer.step()

            optimizer.zero_grad(
                set_to_none=True
            )

            optimizer_step += 1


            if (
                is_main()
                and optimizer_step
                % SAVE_EVERY_OPT_STEPS
                == 0
            ):

                LATEST_DIR.mkdir(
                    parents=True,
                    exist_ok=True,
                )

                model.module.save_pretrained(
                    LATEST_DIR
                )


        if is_main():

            running_jsd = (
                local_loss_sum
                / max(
                    local_examples,
                    1,
                )
            )

            progress.set_postfix(
                jsd=f"{running_jsd:.6f}",
                opt_step=optimizer_step,
            )


    # ========================================================
    # AGGREGATE TRAINING STATISTICS
    # ========================================================

    stats = torch.tensor(
        [
            local_loss_sum,
            float(
                local_examples
            ),
        ],
        dtype=torch.float64,
        device=DEVICE,
    )

    dist.all_reduce(
        stats,
        op=dist.ReduceOp.SUM,
    )

    global_jsd = (
        stats[
            0
        ].item()
        / max(
            stats[
                1
            ].item(),
            1.0,
        )
    )


    # ========================================================
    # SAVE THIS EPOCH
    # ========================================================

    if is_main():

        epoch_dir = (
            ADAPTER_DIR
            / f"epoch_{epoch}"
        )

        epoch_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        model.module.save_pretrained(
            epoch_dir
        )

        print(
            f"Epoch {epoch} training complete | "
            f"mean JSD={global_jsd:.6f}"
        )


    # Both ranks finish training first.
    dist.barrier()


    # ========================================================
    # VALIDATION ON RANK 0
    # ========================================================

    if is_main():

        (
            val_bertscore_f1,
            val_exact_match,
        ) = validate_epoch(
            epoch
        )


        improved = (
            np.isfinite(
                val_bertscore_f1
            )
            and val_bertscore_f1
            > best_bertscore_f1
        )


        if improved:

            best_bertscore_f1 = (
                val_bertscore_f1
            )

            best_epoch = epoch


            if BEST_DIR.exists():
                shutil.rmtree(
                    BEST_DIR
                )


            shutil.copytree(
                ADAPTER_DIR
                / f"epoch_{epoch}",
                BEST_DIR,
            )


            with open(
                BEST_DIR
                / "best_checkpoint.json",
                "w",
            ) as handle:

                json.dump(
                    {
                        "best_epoch":
                            best_epoch,

                        "val_bertscore_f1":
                            best_bertscore_f1,

                        "val_exact_match_accuracy":
                            val_exact_match,

                        "selection_metric":
                            "BERTScore-F1",
                    },
                    handle,
                    indent=2,
                )


        history.append(
            {
                "epoch":
                    epoch,

                "mean_train_jsd":
                    global_jsd,

                "val_bertscore_f1":
                    val_bertscore_f1,

                "val_exact_match_accuracy":
                    val_exact_match,

                "optimizer_steps":
                    optimizer_step,

                "is_best":
                    bool(
                        improved
                    ),
            }
        )


        pd.DataFrame(
            history
        ).to_csv(
            RUN_DIR
            / "training_history.csv",
            index=False,
        )


        with open(
            RUN_DIR
            / f"epoch_{epoch}_metrics.json",
            "w",
        ) as handle:

            json.dump(
                history[
                    -1
                ],
                handle,
                indent=2,
            )


        print(
            f"Epoch {epoch} validation | "
            f"BERTScore-F1="
            f"{val_bertscore_f1:.4f} | "
            f"Exact Match="
            f"{val_exact_match:.4f} | "
            f"best_epoch="
            f"{best_epoch}"
        )


    # Rank 1 waits here while rank 0 performs validation.
    # The process-group timeout is one hour.
    dist.barrier()

    model.train()


# ============================================================
# FINAL SUMMARY
# ============================================================

if is_main():

    if best_epoch is None:
        raise RuntimeError(
            "No valid validation BERTScore-F1 was produced."
        )


    final_metrics = {
        "method":
            "OPSD",

        "model_key":
            MODEL_KEY,

        "model_id":
            MODEL_ID,

        "seed":
            SEED,

        "world_size":
            WORLD_SIZE,

        "epochs":
            EPOCHS,

        "selection_metric":
            "validation BERTScore-F1",

        "bertscore_model":
            BERTSCORE_MODEL,

        "best_epoch":
            int(
                best_epoch
            ),

        "best_val_bertscore_f1":
            float(
                best_bertscore_f1
            ),

        "learning_rate":
            LEARNING_RATE,

        "per_device_batch_size":
            PER_DEVICE_BATCH_SIZE,

        "grad_accum_steps":
            GRAD_ACCUM_STEPS,

        "effective_batch_size":
            (
                PER_DEVICE_BATCH_SIZE
                * WORLD_SIZE
                * GRAD_ACCUM_STEPS
            ),

        "lora_r":
            LORA_R,

        "lora_alpha":
            LORA_ALPHA,

        "distill_temperature":
            DISTILL_TEMPERATURE,

        "fixed_teacher":
            FIXED_TEACHER,

        "jsd_beta":
            JSD_BETA,

        "jsd_token_clip":
            JSD_TOKEN_CLIP,

        "ce_anchor_weight":
            CE_ANCHOR_WEIGHT,

        "train_examples":
            int(
                len(
                    train_df
                )
            ),

        "val_examples":
            int(
                len(
                    val_df
                )
            ),

        "best_adapter_path":
            str(
                BEST_DIR
            ),
    }


    with open(
        RUN_DIR
        / "final_metrics.json",
        "w",
    ) as handle:

        json.dump(
            final_metrics,
            handle,
            indent=2,
        )


    print(
        "\nFinal OPSD selection:"
    )

    print(
        json.dumps(
            final_metrics,
            indent=2,
        )
    )


# Final cleanup is safe because both ranks reach this point.
dist.barrier()

dist.destroy_process_group()
"""

script_path = Path(
    "/kaggle/working/train_opsd_ddp.py"
)

script_path.write_text(
    TRAIN_SCRIPT
)

print(
    "Wrote:",
    script_path
)


Wrote: /kaggle/working/train_opsd_ddp.py


## Export notebook settings to the DDP processes

For an initial smoke test use:

```python
TRAIN_LIMIT = 64
VAL_LIMIT = 64
```

Then switch both back to `None` for the full experiment.


In [6]:
import os

os.environ[
    "KAGGLE_DATASET_SLUG"
] = KAGGLE_DATASET_SLUG

os.environ[
    "MODEL_KEY"
] = MODEL_KEY

os.environ[
    "SEED"
] = str(
    SEED
)

os.environ[
    "EPOCHS"
] = str(
    EPOCHS
)

os.environ[
    "LEARNING_RATE"
] = str(
    LEARNING_RATE
)

os.environ[
    "WEIGHT_DECAY"
] = str(
    WEIGHT_DECAY
)

os.environ[
    "PER_DEVICE_BATCH_SIZE"
] = str(
    PER_DEVICE_BATCH_SIZE
)

os.environ[
    "GRAD_ACCUM_STEPS"
] = str(
    GRAD_ACCUM_STEPS
)

os.environ[
    "MAX_GRAD_NORM"
] = str(
    MAX_GRAD_NORM
)

os.environ[
    "LORA_R"
] = str(
    LORA_R
)

os.environ[
    "LORA_ALPHA"
] = str(
    LORA_ALPHA
)

os.environ[
    "LORA_DROPOUT"
] = str(
    LORA_DROPOUT
)

os.environ[
    "MAX_ROLLOUT_TOKENS"
] = str(
    MAX_ROLLOUT_TOKENS
)

os.environ[
    "MAX_EVAL_TOKENS"
] = str(
    MAX_EVAL_TOKENS
)

os.environ[
    "DISTILL_TEMPERATURE"
] = str(
    DISTILL_TEMPERATURE
)

os.environ[
    "FIXED_TEACHER"
] = (
    "1"
    if FIXED_TEACHER
    else "0"
)

os.environ[
    "JSD_BETA"
] = str(
    JSD_BETA
)

os.environ[
    "JSD_TOKEN_CLIP"
] = str(
    JSD_TOKEN_CLIP
)

os.environ[
    "CE_ANCHOR_WEIGHT"
] = str(
    CE_ANCHOR_WEIGHT
)

os.environ[
    "SAVE_EVERY_OPT_STEPS"
] = str(
    SAVE_EVERY_OPT_STEPS
)

os.environ[
    "BERTSCORE_MODEL"
] = BERTSCORE_MODEL

os.environ[
    "BERTSCORE_BATCH_SIZE"
] = str(
    BERTSCORE_BATCH_SIZE
)

os.environ[
    "TRAIN_LIMIT"
] = (
    ""
    if TRAIN_LIMIT is None
    else str(
        TRAIN_LIMIT
    )
)

os.environ[
    "VAL_LIMIT"
] = (
    ""
    if VAL_LIMIT is None
    else str(
        VAL_LIMIT
    )
)

os.environ[
    "TOKENIZERS_PARALLELISM"
] = "false"

print(
    "DDP environment configured."
)


DDP environment configured.


## Launch training on both T4 GPUs

You should see:

```text
World size: 2
```

near the beginning of the output. That confirms both GPUs are participating.


In [ ]:
!torchrun --standalone --nproc_per_node=2 /kaggle/working/train_opsd_ddp.py

W1025 06:55:29.899000 137 torch/distributed/run.py:982] 
W1025 06:55:29.899000 137 torch/distributed/run.py:982] *****************************************
W1025 06:55:29.899000 137 torch/distributed/run.py:982] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1025 06:55:29.899000 137 torch/distributed/run.py:982] *****************************************
World size: 2
Train examples: 3507
Validation examples: 758
Effective batch size: 8
Loading model: HuggingFaceTB/SmolVLM-500M-Instruct
processor_config.json: 100%|█████████████████| 68.0/68.0 [00:00<00:00, 309kB/s]
preprocessor_config.json: 100%|███████████████| 486/486 [00:00<00:00, 2.58MB/s]
config.json: 100%|████████████████████████| 7.34k/7.34k [00:00<00:00, 18.1MB/s]
tokenizer_config.json: 100%|██████████████| 28.2k/28.2k [00:00<00:00, 74.4MB/s]
vocab.json: 100%|█████████

## Inspect saved results

The primary validation metric is `val_bertscore_f1`; normalised exact match is retained as `val_exact_match_accuracy`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

RUN_DIR = (
    Path("/kaggle/working")
    / "results"
    / "training"
    / "opsd"
    / MODEL_KEY
    / f"seed{SEED}"
)

ADAPTER_DIR = (
    Path("/kaggle/working")
    / "models"
    / "opsd"
    / MODEL_KEY
    / f"seed{SEED}"
)

BEST_DIR = (
    ADAPTER_DIR
    / "best"
)

print(
    "Run directory:",
    RUN_DIR,
)

print(
    "Adapter root:",
    ADAPTER_DIR,
)

print(
    "Best adapter:",
    BEST_DIR,
)


history_path = (
    RUN_DIR
    / "training_history.csv"
)

metrics_path = (
    RUN_DIR
    / "final_metrics.json"
)

best_checkpoint_path = (
    BEST_DIR
    / "best_checkpoint.json"
)


if history_path.is_file():

    print(
        "\nTraining + validation history:"
    )

    display(
        pd.read_csv(
            history_path
        )
    )


if metrics_path.is_file():

    print(
        "\nFinal metrics:"
    )

    with open(
        metrics_path,
        "r",
    ) as handle:

        metrics = json.load(
            handle
        )

    print(
        json.dumps(
            metrics,
            indent=2,
        )
    )


if best_checkpoint_path.is_file():

    print(
        "\nBest checkpoint:"
    )

    with open(
        best_checkpoint_path,
        "r",
    ) as handle:

        best_info = json.load(
            handle
        )

    print(
        json.dumps(
            best_info,
            indent=2,
        )
    )


## Notes

### Output structure

After a successful 3-epoch run:

```text
/kaggle/working/
├── models/
│   └── opsd/
│       └── <model>/
│           └── seed42/
│               ├── epoch_1/
│               ├── epoch_2/
│               ├── epoch_3/
│               ├── latest/
│               └── best/
│                   ├── adapter_config.json
│                   ├── adapter_model.safetensors
│                   └── best_checkpoint.json
│
└── results/
    └── training/
        └── opsd/
            └── <model>/
                └── seed42/
                    ├── training_history.csv
                    ├── epoch_1_metrics.json
                    ├── epoch_2_metrics.json
                    ├── epoch_3_metrics.json
                    ├── val_epoch1_predictions.csv
                    ├── val_epoch1_predictions_scored.csv
                    ├── ...
                    └── final_metrics.json
```

For the later local comparison notebook, preserve both `/kaggle/working/models/opsd/...` and `/kaggle/working/results/training/opsd/...`.
